# Overview
This notebook explores the use of MalSkills to analyze agent skills from the GitSkills dataset for potential security issues.

The goal is to automate the analysis of the 100 SKILL.md files exported in 01_prepare_skills.ipynb, collect MalSkill's findings, and organize the results for further investigation.


## Objectives
- Understand how MalSkills works and what security-related information it produces
- Develop a workflow for automatically analyzing multiple skills
- Collect and organize the generated reports and JSON files.
- Identify recurring security issues and potentially problematic behaviours.
- Investigate how these findings contribute to RQ1.

## Scope
For this experiment, only the exported SKILL.md files will be analyzed. Supporting files, such as scripts and references, are excluded. This allows us to explore what security-relevant information malSkills can identify from SKILL.md alone.

Findings will be treated as potential issues requiring further investigatino rather than confirmed vulnerabilities.

# MalSkill CLI Commands
```bash
malskills --help
```
This displays the available commands provided by MalSkills.
| Command | Purpose |
|---|---|
| `analyze-skill` | Analyze an individual skill |
| `scan-corpus` | Scan a collection of skills |
| `render-report` | Generate a report from analysis results |
| `rules` | Work with detection rules |
| `show-llm-config` | Display LLM configuration |
| `build-benchmark-index` | Prepare benchmark indexing |
| `run-eval` | Run evaluations |

# Environment Setup
This notebook uses Python's subprocess module to invoke MalSkills, a CLI tool for analyzing agent skills.

For this experiment, MalSkills was installed in an Ubuntu environment using Windows Subsytem for Linux (WSL), since the development machine runs Windows.

The Jupyter notebook runs in a Windows Python environment and communicates with the MalSkills installation via WSL.

This configuration is specific to the current development environment and is not necessarily required by MalSkills.

Users running macOS or Linux may be able to invoke MalSkills directly through a local Python environment. Platform compatibility should be verified against MalSkill's installation requirements.

For this experiment, the MalSkills repository should be cloned into the same parent directory as SkillGuard, rather than inside the SkillGuard directory.

## Path Configuration
The following cell locates the SkillGuard and MalSkills directories relative to the notebook's working directory. This avoids hardcoding user-specific file paths and allows the project to be relocated, provided both repositories maintain the expected directory structure.

In [1]:
from pathlib import Path

# Locate project directories relative to the notebook
skillguard_root = Path.cwd().resolve().parent
malskills_root = skillguard_root.parent / "MalSkills"

print("SkillGuard:", skillguard_root)
print("MalSkills:", malskills_root)

assert skillguard_root.is_dir(), "SkillGuard directory not found."
assert malskills_root.is_dir(), "MalSkills directory not found."

print("Both project directories found.")

SkillGuard: C:\Users\ziyoo\Documents\year4\SkillGuard
MalSkills: C:\Users\ziyoo\Documents\year4\MalSkills
Both project directories found.


## MalSkills Execution Configuration

MalSkills is accessed through Python's `subprocess` module, which allows the notebook to execute external command-line programs.

For the current Windows environment, commands are passed to the Ubuntu installation through WSL. Since Windows and Linux use different filesystem path formats, the repository paths must first be converted using `wslpath`.

The following cell converts the MalSkills directory into a Linux-compatible path and verifies that the installed CLI is accessible.

This configuration is specific to Windows with WSL. Other operating systems may require different execution commands.

In [2]:
import subprocess

# Convert the Windows path to a Linux path
result = subprocess.run(
    ["wsl", "-d", "Ubuntu", "--", "wslpath", "-a", malskills_root.as_posix()],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

malskills_linux_path = result.stdout.strip()

# Run MalSkills from its virtual environment
result = subprocess.run(
    [
        "wsl", "-d", "Ubuntu", "--",
        f"{malskills_linux_path}/.venv-wsl/bin/malskills",
        "--help"
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

Return code: 0
Output: /mnt/c/Users/ziyoo/Documents/year4/MalSkills

Error: 
Return code: 0
Output: usage: malskills [-h]
                 {analyze-skill,build-benchmark-index,scan-corpus,run-eval,render-report,show-llm-config,rules}
                 ...

positional arguments:
  {analyze-skill,build-benchmark-index,scan-corpus,run-eval,render-report,show-llm-config,rules}

options:
  -h, --help            show this help message and exit

Error: 


# Investigating `scan-corpus`
The `scan-corpus` command appears to support automated analysis of multiple skills. To inspect its available arguments, run:
```bash
malskills scan-corpus --help
```

The command requires three arguments and a specific folder structure:
| Argument | Purpose |
|---|---|
| `corpus-root` | Specifies the directory containing the skills to analyze |
| `output` | Specifies where analysis results will be saved |
| `--sample-size` | Specifies the # of skills to sample |

`<courpus-root>/<source>/<package>/SKILL.md`

There are additional options such as `--seed` (a pseudorandom number), `--workers`(# of processes responsible for completing a task), and `--profile`(the type of analysis to perform).

Note: MalSkills has two profiles. 
`static` analysis: Examines skill content using static extraction and formal graph-based rules, without the LLM stages.
`full` analysis: Includes static analysis and any LLM-assisted staged enabled in the TOML configuration

After inspecting `malskills/corpus_scan.py`, I found that the corpus scanner:
1. Identifies source directories within the specified corpus root.
2. Identifies package directories with each source.
3. Recursively searches each package for a `SKILL.md` file.
4. Removes skills with identical content by default.

The existing exported direction structure is compatible with these requirements:
```text
SkillGuard/
└── exported_skills/
    └── skill_XXXX/
        └── SKILL.md
```
`SkillGuard/` will be used as the corpus root.


## Running the Initial Corpus Scan

### Preparing and Verifying Exported Skills
Before running the corpus scan, we verofy that 100 SKILL.md files prepared in `01_prepare_skills.ipynb` are present and accessible.

The following cell:
1. Locates the `exported_skills/` directory within SkillGuard.
2. Identifies `SKILL.md` files within the numbered `skill_xxxx/` directories.
3. Counts the files and displays the path of the first file for inspection
4. Uses an assertion to confirm that exactly 100 files were found.

This verification ensures that the expected sample is available before proceeding with an automated analysis. It does not examine the contents of the files or execute MalSkills.

Expected result: 100 `SKILL.md` files

In [3]:
# Locate the SkillGuard corpus in WSL
# Verify the exported skill files before scanning
result = subprocess.run(
    ["wsl", "-d", "Ubuntu", "--", "wslpath", "-a", skillguard_root.as_posix()],
    capture_output=True,
    text=True,
    check=True
)

skillguard_linux_path = result.stdout.strip()

print("Corpus root:", skillguard_linux_path)
print("MalSkills:", malskills_linux_path)

# Verify the exported skill files before scanning
export_dir = skillguard_root / "exported_skills"
skill_files = sorted(export_dir.glob("skill_*/SKILL.md"))

print("Export directory:", export_dir)
print("Number of SKILL.md files:", len(skill_files))
print("First skill:", skill_files[0] if skill_files else "None")

assert len(skill_files) == 100, "Expected 100 exported skills."

Corpus root: /mnt/c/Users/ziyoo/Documents/year4/SkillGuard
MalSkills: /mnt/c/Users/ziyoo/Documents/year4/MalSkills
Export directory: C:\Users\ziyoo\Documents\year4\SkillGuard\exported_skills
Number of SKILL.md files: 100
First skill: C:\Users\ziyoo\Documents\year4\SkillGuard\exported_skills\skill_0000\SKILL.md


### Configuring the Corpus Scan
Prior to executing MalSkills, we define the input corpus, output directory, and analysis settings.

The initial scan will use the following configuration:
- **Input:** The 100 exported `SKILL.md` files in `exported_skills/`
- **Output:** `results/malskills/test_scan/`
- **Analysis profile:** `static`, which excludes LLM-assisted analysis.
- **Sample size:** 100 skills

The following cell prepares the output directory and defines the scan settings. It does not execute the scan.

In [11]:
# Configure the initial corpus scan
# Change the output folder name for each experiment to avoid overwriting previous results.
output_dir = skillguard_root / "results" / "malskills" / "test_scan_100"

# !!! Adjust the sample as needed (1 for testing, 100 for the full sample)
scan_sample_size = 100 #
scan_profile = "static"

print("Corpus root:", skillguard_linux_path)
print("Output directory:", output_dir)
print("Sample size:", scan_sample_size)
print("Analysis profile:", scan_profile)

Corpus root: /mnt/c/Users/ziyoo/Documents/year4/SkillGuard
Output directory: C:\Users\ziyoo\Documents\year4\SkillGuard\results\malskills\test_scan_100
Sample size: 100
Analysis profile: static


### Executing the Initial Corpus Scan

This cell runs MalSkills on the exported skill corpus using the static analysis profile. The results will be saved to `results/malskills/initial_scan/`.

In [12]:
# Convert the output directory to a WSL-compatible path
result = subprocess.run(
    ["wsl", "-d", "Ubuntu", "--", "wslpath", "-a", output_dir.as_posix()],
    capture_output=True,
    text=True,
    check=True
)

output_linux_path = result.stdout.strip()

# Execute the MalSkills corpus scan
result = subprocess.run(
    [
        "wsl", "-d", "Ubuntu", "--",
        f"{malskills_linux_path}/.venv-wsl/bin/malskills",
        "scan-corpus",
        "--corpus-root", skillguard_linux_path,
        "--output", output_linux_path,
        "--sample-size", str(scan_sample_size),
        "--profile", scan_profile,
        "--source", "exported_skills",      # Restrict discovery to our exported_skills folder
        "--seed", "1337",                   # Random seed for reproducible sampling
        "--workers", "4",                   # Number of parallel workers processing skills
        "--retain", "all",                  # Preserve detailed reports for all scanned skills, including benign ones
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

Return code: 0
Output: scanned=100 malicious=0 benign=100 errors=0 report=/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/results/malskills/test_scan_100/scan_summary.json

Error: 02:32:37 | INFO    | START sample=100 pending=100 resumed=0 workers=4 profile=static retain=all output=/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/results/malskills/test_scan_100
02:32:40 | SUCCESS | [1/100] OK verdict=benign time=1.6s findings=0 ssos=0 patterns=none id=exported_skills::skill_0001
02:32:40 | SUCCESS | [2/100] OK verdict=benign time=1.6s findings=0 ssos=0 patterns=none id=exported_skills::skill_0000
02:32:41 | SUCCESS | [3/100] OK verdict=benign time=1.9s findings=0 ssos=0 patterns=none id=exported_skills::skill_0002
02:32:41 | SUCCESS | [4/100] OK verdict=benign time=2.1s findings=0 ssos=0 patterns=none id=exported_skills::skill_0003
02:32:43 | SUCCESS | [5/100] OK verdict=benign time=2.1s findings=0 ssos=0 patterns=none id=exported_skills::skill_0005
02:32:43 | SUCCESS | [6/100] OK verdict=b

### Manual Inspection: `skill_0000`

**Skill:** Configuring Identity-Aware Proxy with Google IAP

**MalSkills Results:**
- Verdict: Benign
- Artifacts extracted: 7 (1 Markdown, 5 Shell, 1 Python)
- Security findings: 0

**Manually Identified Behaviors:**

| Behavior | Evidence | Observed |
|---|---|---|
| Command execution | `gcloud services enable iap.googleapis.com` | Yes |
| Network access | `authed_session.get(IAP_URL)` | Yes |
| Credential access | `google.auth.default()` | Yes |

**Observation:**

MalSkills classified the skill as benign despite the presence of three security-relevant behaviors in its instructions and code examples.

These behaviors are not necessarily malicious. This suggests that MalSkills' maliciousness detection and manual identification of security-relevant behaviors measure different things.

**Note:** The behaviors were identified in `SKILL.md`; no actual execution was observed.